# 🌾 Hagridculture: Advanced Irrigation Prediction
## Target: Top 100 Leaderboard

**Strategy:**
- Comprehensive feature engineering (60+ features)
- 3-model ensemble (XGBoost + LightGBM + CatBoost)
- Balanced accuracy optimization
- Post-processing with threshold optimization

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("🌾 HAGRIDCULTURE - Advanced Irrigation Prediction")
print("="*80)

In [ ]:
# LOAD DATA
print("\n📊 Loading data...")
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

# Auto-detect target column
target_candidates = [c for c in train.columns if c.lower() in ['irrigation_need', 'target', 'label']]
if not target_candidates:
    target_candidates = [train.columns[-1]]
    print(f'  ⚠️ Auto-detected target column: {target_candidates[0]}')
target_col = target_candidates[0]

# Rename to standard name
if target_col != 'Irrigation_Need':
    train = train.rename(columns={target_col: 'Irrigation_Need'})

# Auto-detect ID column
id_candidates = [c for c in test.columns if c.lower() in ['id', 'row_id', 'sample_id']]
if not id_candidates:
    id_candidates = [test.columns[0]]
id_col = id_candidates[0]

print(f"  Train shape: {train.shape}")
print(f"  Test shape: {test.shape}")
print(f"  Target distribution:")
print(train['Irrigation_Need'].value_counts())
print(f"  ID column: {id_col}")

In [ ]:
# FEATURE ENGINEERING
print("\n🔧 Engineering features...")

def create_features(df, train_df=None):
    df = df.copy()
    
    categorical_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
                       'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
    
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
    
    # Interaction features (15)
    df['Soil_pH_Soil_Moisture'] = df['Soil_pH'] * df['Soil_Moisture']
    df['Temperature_Humidity'] = df['Temperature_C'] * df['Humidity']
    df['Rainfall_Soil_Moisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
    df['Sunlight_Temperature'] = df['Sunlight_Hours'] * df['Temperature_C']
    df['Wind_Humidity'] = df['Wind_Speed_kmh'] * df['Humidity']
    df['Organic_Temperature'] = df['Organic_Carbon'] * df['Temperature_C']
    df['EC_Moisture'] = df['Electrical_Conductivity'] * df['Soil_Moisture']
    df['Rainfall_Humidity'] = df['Rainfall_mm'] * df['Humidity']
    df['Wind_Temperature'] = df['Wind_Speed_kmh'] * df['Temperature_C']
    df['Sunlight_Humidity'] = df['Sunlight_Hours'] * df['Humidity']
    df['PrevIrrigation_Temperature'] = df['Previous_Irrigation_mm'] * df['Temperature_C']
    df['PrevIrrigation_Humidity'] = df['Previous_Irrigation_mm'] * df['Humidity']
    df['Soil_pH_Temperature'] = df['Soil_pH'] * df['Temperature_C']
    df['EC_Rainfall'] = df['Electrical_Conductivity'] * df['Rainfall_mm']
    df['Organic_Humidity'] = df['Organic_Carbon'] * df['Humidity']
    
    # Ratio features (10)
    df['Organic_to_Soil'] = df['Organic_Carbon'] / (df['Soil_Moisture'] + 1e-6)
    df['EC_to_Soil_pH'] = df['Electrical_Conductivity'] / (df['Soil_pH'] + 1e-6)
    df['Rainfall_per_Hectare'] = df['Rainfall_mm'] / (df['Field_Area_hectare'] + 1e-6)
    df['Prev_Irrigation_per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 1e-6)
    df['Temperature_Humidity_Ratio'] = df['Temperature_C'] / (df['Humidity'] + 1e-6)
    df['Rainfall_Humidity_Ratio'] = df['Rainfall_mm'] / (df['Humidity'] + 1e-6)
    df['Sunlight_Rainfall_Ratio'] = df['Sunlight_Hours'] / (df['Rainfall_mm'] + 1e-6)
    df['Wind_Rainfall_Ratio'] = df['Wind_Speed_kmh'] / (df['Rainfall_mm'] + 1e-6)
    df['EC_Organic_Ratio'] = df['Electrical_Conductivity'] / (df['Organic_Carbon'] + 1e-6)
    df['Moisture_FieldArea_Ratio'] = df['Soil_Moisture'] / (df['Field_Area_hectare'] + 1e-6)
    
    # Polynomial & transform features (8)
    df['Soil_pH_sq'] = df['Soil_pH'] ** 2
    df['Moisture_sq'] = df['Soil_Moisture'] ** 2
    df['Temperature_sq'] = df['Temperature_C'] ** 2
    df['Humidity_sq'] = df['Humidity'] ** 2
    df['Rainfall_log'] = np.log1p(df['Rainfall_mm'])
    df['Prev_Irrigation_log'] = np.log1p(df['Previous_Irrigation_mm'])
    df['Humidity_log'] = np.log1p(df['Humidity'])
    df['Organic_log'] = np.log1p(df['Organic_Carbon'])
    
    # Domain-specific features (8)
    df['Water_Stress'] = (df['Temperature_C'] * df['Wind_Speed_kmh']) / (df['Humidity'] + df['Soil_Moisture'] + 1e-6)
    df['Irrigation_Efficiency'] = df['Previous_Irrigation_mm'] / (df['Rainfall_mm'] + 1e-6)
    df['Crop_Water_Demand'] = df['Sunlight_Hours'] * df['Temperature_C'] / (df['Humidity'] + 1e-6)
    df['Evapotranspiration_Proxy'] = (df['Temperature_C'] * 0.7 + df['Sunlight_Hours'] * 0.3) * df['Wind_Speed_kmh'] / (df['Humidity'] + 1e-6)
    df['Soil_Water_Retention'] = df['Organic_Carbon'] * df['Soil_Moisture'] / (df['Soil_pH'] + 1e-6)
    df['Drought_Index'] = (df['Rainfall_mm'] + df['Previous_Irrigation_mm']) / (df['Temperature_C'] * df['Wind_Speed_kmh'] + 1e-6)
    df['GDD_Proxy'] = np.maximum(df['Temperature_C'] - 10, 0) * df['Sunlight_Hours']
    df['VPD_Proxy'] = df['Temperature_C'] * (100 - df['Humidity']) / 100
    
    # Binned features
    if len(df) > 10000:
        df['Soil_Moisture_bin'] = pd.qcut(df['Soil_Moisture'], q=10, labels=False, duplicates='drop')
        df['Rainfall_bin'] = pd.qcut(df['Rainfall_mm'], q=10, labels=False, duplicates='drop')
        df['Temperature_bin'] = pd.qcut(df['Temperature_C'], q=10, labels=False, duplicates='drop')
        df['Humidity_bin'] = pd.qcut(df['Humidity'], q=10, labels=False, duplicates='drop')
    
    # Cross features
    df['Crop_Soil'] = df['Crop_Type'].astype(str) + '_' + df['Soil_Type'].astype(str)
    df['Crop_Region'] = df['Crop_Type'].astype(str) + '_' + df['Region'].astype(str)
    df['Crop_Season'] = df['Crop_Type'].astype(str) + '_' + df['Season'].astype(str)
    df['Soil_Region'] = df['Soil_Type'].astype(str) + '_' + df['Region'].astype(str)
    
    for col in ['Crop_Soil', 'Crop_Region', 'Crop_Season', 'Soil_Region']:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
    
    # Target encoding
    if train_df is not None:
        target_groups = ['Crop_Type', 'Soil_Type', 'Region', 'Season', 'Irrigation_Type']
        for group_col in target_groups:
            if 'Irrigation_Need' in train_df.columns:
                temp_df = train_df.copy()
                temp_df['target_num'] = temp_df['Irrigation_Need'].map({'Low': 0, 'Medium': 1, 'High': 2})
                group_means = temp_df.groupby(group_col)['target_num'].mean()
                df[f'{group_col}_target_mean'] = df[group_col].map(group_means)
                group_high_prob = (temp_df['Irrigation_Need'] == 'High').groupby(temp_df[group_col]).mean()
                df[f'{group_col}_high_prob'] = df[group_col].map(group_high_prob)
    
    return df, label_encoders

train['is_train'] = True
test['is_train'] = False

combined = pd.concat([train, test], axis=0, ignore_index=True)
combined, label_encoders = create_features(combined, train_df=train)

train_feat = combined[combined['is_train'] == True].copy()
test_feat = combined[combined['is_train'] == False].copy()

print(f"  Features created: {train_feat.shape[1]}")
print(f"  Train shape: {train_feat.shape}")
print(f"  Test shape: {test_feat.shape}")

In [ ]:
# PREPARE DATA
print("\n📋 Preparing data...")

drop_cols = ['is_train', id_col]
if 'Irrigation_Need' in test_feat.columns:
    test_feat = test_feat.drop('Irrigation_Need', axis=1)

train_feat = train_feat.drop(columns=[c for c in drop_cols if c in train_feat.columns])
test_feat = test_feat.drop(columns=[c for c in drop_cols if c in test_feat.columns])

target_map = {'Low': 0, 'Medium': 1, 'High': 2}
target_inverse = {0: 'Low', 1: 'Medium', 2: 'High'}
y = train['Irrigation_Need'].map(target_map).values

X = train_feat.drop('Irrigation_Need', axis=1, errors='ignore')
X_test = test_feat.copy()

common_cols = [c for c in X.columns if c in X_test.columns]
X = X[common_cols]
X_test = X_test[common_cols]

X = X.fillna(0)
X_test = X_test.fillna(0)

print(f"  Features: {X.shape[1]}")
print(f"  Classes: {np.unique(y)}")
print(f"  Target distribution: {pd.Series(y).value_counts().to_dict()}")

In [ ]:
# MODEL CONFIGURATION
print("\n⚙️ Configuring models...")

N_SPLITS = 5
RANDOM_SEED = 42

xgb_params = {
    'n_estimators': 1500,
    'max_depth': 7,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'min_child_weight': 10,
    'gamma': 0.2,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'random_state': RANDOM_SEED,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'verbosity': 0
}

lgb_params = {
    'n_estimators': 1500,
    'max_depth': 7,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'min_child_samples': 30,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'class_weight': 'balanced',
    'random_state': RANDOM_SEED,
    'verbosity': -1
}

cat_params = {
    'iterations': 1500,
    'depth': 7,
    'learning_rate': 0.03,
    'l2_leaf_reg': 2.0,
    'random_seed': RANDOM_SEED,
    'verbose': 0,
    'task_type': 'CPU'
}

print("  ✓ XGBoost: 1500 estimators, lr=0.03, depth=7")
print("  ✓ LightGBM: 1500 estimators, lr=0.03, depth=7, balanced")
print("  ✓ CatBoost: 1500 iterations, lr=0.03, depth=7")

In [ ]:
# TRAIN ENSEMBLE MODELS (with memory management)
import gc

print("\n🚀 Training ensemble models...")
print(f"  Dataset: {len(X):,} samples, {X.shape[1]} features")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

oof_xgb = np.zeros((len(X), 3))
oof_lgb = np.zeros((len(X), 3))
oof_cat = np.zeros((len(X), 3))

test_xgb = np.zeros((len(X_test), 3))
test_lgb = np.zeros((len(X_test), 3))
test_cat = np.zeros((len(X_test), 3))

xgb_importances = np.zeros(X.shape[1])
lgb_importances = np.zeros(X.shape[1])
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/{N_SPLITS}")
    print(f"{'='*60}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # XGBoost
    print("  Training XGBoost...", end=' ')
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)
    xgb_importances += xgb_model.feature_importances_
    xgb_pred = np.argmax(oof_xgb[val_idx], axis=1)
    xgb_acc = balanced_accuracy_score(y_val, xgb_pred)
    print(f"Balanced Acc: {xgb_acc:.6f}")
    test_xgb += xgb_model.predict_proba(X_test) / N_SPLITS
    del xgb_model, X_train, X_val, y_train, y_val
    gc.collect()
    print("    ✓ Memory freed")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # LightGBM
    print("  Training LightGBM...", end=' ')
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                 callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(period=0)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)
    lgb_importances += lgb_model.feature_importances_
    lgb_pred = np.argmax(oof_lgb[val_idx], axis=1)
    lgb_acc = balanced_accuracy_score(y_val, lgb_pred)
    print(f"Balanced Acc: {lgb_acc:.6f}")
    test_lgb += lgb_model.predict_proba(X_test) / N_SPLITS
    del lgb_model, X_train, X_val, y_train, y_val
    gc.collect()
    print("    ✓ Memory freed")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # CatBoost
    print("  Training CatBoost...", end=' ')
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val),
                 early_stopping_rounds=150, verbose=False)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)
    cat_pred = np.argmax(oof_cat[val_idx], axis=1)
    cat_acc = balanced_accuracy_score(y_val, cat_pred)
    print(f"Balanced Acc: {cat_acc:.6f}")
    test_cat += cat_model.predict_proba(X_test) / N_SPLITS
    del cat_model, X_train, X_val, y_train, y_val
    gc.collect()
    print("    ✓ Memory freed")
    
    fold_scores.append({'fold': fold + 1, 'xgb': xgb_acc, 'lgb': lgb_acc, 'cat': cat_acc})
    print(f"\n  ✅ Fold {fold+1} complete - All memory released")

xgb_importances /= N_SPLITS
lgb_importances /= N_SPLITS

print(f"\n{'='*60}")
print("✅ ALL FOLDS COMPLETE - Training Complete!")
print(f"{'='*60}")

print(f"\n📊 Fold-by-Fold Performance:")
for fs in fold_scores:
    print(f"  Fold {fs['fold']}: XGB={fs['xgb']:.6f} LGB={fs['lgb']:.6f} CAT={fs['cat']:.6f}")

del fold_scores
gc.collect()

In [ ]:
# EVALUATE OOF PERFORMANCE
print("\n📊 Evaluating OOF performance...")

oof_xgb_pred = np.argmax(oof_xgb, axis=1)
oof_lgb_pred = np.argmax(oof_lgb, axis=1)
oof_cat_pred = np.argmax(oof_cat, axis=1)

print(f"\nIndividual Models (Balanced Accuracy):")
print(f"  XGBoost:  {balanced_accuracy_score(y, oof_xgb_pred):.6f}")
print(f"  LightGBM: {balanced_accuracy_score(y, oof_lgb_pred):.6f}")
print(f"  CatBoost: {balanced_accuracy_score(y, oof_cat_pred):.6f}")

ensemble_all = (oof_xgb + oof_lgb + oof_cat) / 3
print(f"  XGB+LGB+CAT (equal): {balanced_accuracy_score(y, np.argmax(ensemble_all, axis=1)):.6f}")

ensemble_xgb_lgb = (oof_xgb + oof_lgb) / 2
print(f"  XGB+LGB: {balanced_accuracy_score(y, np.argmax(ensemble_xgb_lgb, axis=1)):.6f}")

ensemble_xgb_cat = (oof_xgb + oof_cat) / 2
print(f"  XGB+CAT: {balanced_accuracy_score(y, np.argmax(ensemble_xgb_cat, axis=1)):.6f}")

ensemble_lgb_cat = (oof_lgb + oof_cat) / 2
print(f"  LGB+CAT: {balanced_accuracy_score(y, np.argmax(ensemble_lgb_cat, axis=1)):.6f}")

ensembles = {
    'XGB+LGB+CAT': ensemble_all,
    'XGB+LGB': ensemble_xgb_lgb,
    'XGB+CAT': ensemble_xgb_cat,
    'LGB+CAT': ensemble_lgb_cat
}

best_name = None
best_score = 0
best_ensemble = None

for name, ens in ensembles.items():
    score = balanced_accuracy_score(y, np.argmax(ens, axis=1))
    if score > best_score:
        best_score = score
        best_name = name
        best_ensemble = ens

print(f"\n🏆 Best Ensemble: {best_name} with {best_score:.6f}")

In [ ]:
# THRESHOLD OPTIMIZATION
print("\n🔍 Optimizing prediction thresholds...")

from scipy.optimize import minimize
import time

def optimize_thresholds(probs, y_true, time_limit=90):
    def objective(thresholds):
        preds = np.ones(len(probs), dtype=int) * -1
        for i in range(probs.shape[1]):
            if i < len(thresholds):
                mask = probs[:, i] > thresholds[i]
                preds[mask] = i
        unassigned = preds == -1
        if unassigned.any():
            preds[unassigned] = np.argmax(probs[unassigned], axis=1)
        return -balanced_accuracy_score(y_true, preds)
    
    best_score = 0
    best_thresholds = None
    start_time = time.time()
    
    for attempt in range(10):
        if time.time() - start_time > time_limit:
            break
        x0 = np.random.uniform(0.2, 0.5, size=probs.shape[1]-1)
        try:
            result = minimize(objective, x0, method='Nelder-Mead', options={'maxiter': 1000})
            if -result.fun > best_score:
                best_score = -result.fun
                best_thresholds = result.x
        except:
            continue
    
    return best_thresholds, best_score

thresholds, opt_score = optimize_thresholds(best_ensemble, y, time_limit=90)

if thresholds is not None and opt_score > best_score:
    print(f"  ✓ Optimized: {opt_score:.6f} (+{opt_score - best_score:.6f})")
    test_ensemble = (test_xgb + test_lgb + test_cat) / 3 if best_name == 'XGB+LGB+CAT' else \
                   (test_xgb + test_lgb) / 2 if best_name == 'XGB+LGB' else \
                   (test_xgb + test_cat) / 2 if best_name == 'XGB+CAT' else \
                   (test_lgb + test_cat) / 2
else:
    print(f"  ⚠️ No improvement (keeping {best_score:.6f})")
    thresholds = None
    test_ensemble = (test_xgb + test_lgb + test_cat) / 3 if best_name == 'XGB+LGB+CAT' else \
                   (test_xgb + test_lgb) / 2 if best_name == 'XGB+LGB' else \
                   (test_xgb + test_cat) / 2 if best_name == 'XGB+CAT' else \
                   (test_lgb + test_cat) / 2

del optimize_thresholds
gc.collect()

In [ ]:
# GENERATE SUBMISSION
print("\n📝 Generating submission...")

if thresholds is not None:
    test_pred = np.ones(len(test_ensemble), dtype=int) * -1
    for i in range(len(thresholds)):
        mask = test_ensemble[:, i] > thresholds[i]
        test_pred[mask] = i
    unassigned = test_pred == -1
    if unassigned.any():
        test_pred[unassigned] = np.argmax(test_ensemble[unassigned], axis=1)
else:
    test_pred = np.argmax(test_ensemble, axis=1)

test_labels = np.array([target_inverse[p] for p in test_pred])

submission = pd.DataFrame({
    id_col: test[id_col],
    'Irrigation_Need': test_labels
})

print(f"  Submission shape: {submission.shape}")
print(f"  Prediction distribution:")
print(submission['Irrigation_Need'].value_counts())

submission.to_csv('submission.csv', index=False)
print(f"\n✅ Submission saved to 'submission.csv'")

del test_pred, test_labels, test_ensemble
gc.collect()
print("💾 Memory freed")

In [ ]:
# FINAL SUMMARY
print("\n" + "="*80)
print("🎉 HAGRIDCULTURE - COMPLETE!")
print("="*80)
print(f"\n Best Ensemble: {best_name}")
print(f" CV Balanced Accuracy: {best_score:.6f}")
print(f"\n This should place you in the TOP 100!")
print("="*80)